In [ ]:
!pip install torchcodec soundfile

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os

# ================= 配置区域 =================
LABELS = ['yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go', 'unknown']
# 在 Colab 里，直接上传的文件通常就在 /content/ 目录下，或者是当前目录
MODEL_PATH = './voice_model.pth'

# ================= 模型定义 (保持不变) =================
class M5(nn.Module):
    def __init__(self, n_input=1, n_output=11, stride=16, n_channel=32):
        super().__init__()
        self.conv1 = nn.Conv1d(n_input, n_channel, kernel_size=80, stride=stride)
        self.bn1 = nn.BatchNorm1d(n_channel)
        self.pool1 = nn.MaxPool1d(4)
        self.conv2 = nn.Conv1d(n_channel, n_channel, kernel_size=3)
        self.bn2 = nn.BatchNorm1d(n_channel)
        self.pool2 = nn.MaxPool1d(4)
        self.conv3 = nn.Conv1d(n_channel, 2 * n_channel, kernel_size=3)
        self.bn3 = nn.BatchNorm1d(2 * n_channel)
        self.pool3 = nn.MaxPool1d(4)
        self.conv4 = nn.Conv1d(2 * n_channel, 2 * n_channel, kernel_size=3)
        self.bn4 = nn.BatchNorm1d(2 * n_channel)
        self.pool4 = nn.MaxPool1d(4)
        self.fc1 = nn.Linear(2 * n_channel, n_output)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(self.bn1(x))
        x = self.pool1(x)
        x = self.conv2(x)
        x = F.relu(self.bn2(x))
        x = self.pool2(x)
        x = self.conv3(x)
        x = F.relu(self.bn3(x))
        x = self.pool3(x)
        x = self.conv4(x)
        x = F.relu(self.bn4(x))
        x = self.pool4(x)
        x = F.avg_pool1d(x, x.shape[-1])
        x = x.permute(0, 2, 1)
        x = self.fc1(x)
        return F.log_softmax(x, dim=2).squeeze(1)

# ================= 演示主程序 =================
def run_online_demo():
    print(" 正在 Colab 云端初始化语音助手...")

    # 1. 检查文件是否存在
    if not os.path.exists(MODEL_PATH):
        print(f" 错误：没找到 {MODEL_PATH}")
        print("请务必点击左侧文件夹图标，把 voice_model.pth 拖进去！")
        return

    # 2. 加载模型
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f" 运行环境: {device}")

    model = M5(n_output=len(LABELS))
    model.to(device)

    try:
        # 加载权重
        state_dict = torch.load(MODEL_PATH, map_location=device)
        model.load_state_dict(state_dict)
        print(" 模型加载成功！参数已读取。")
    except Exception as e:
        print(f" 模型加载失败: {e}")
        return

    # 3. 模拟测试 (生成随机噪音数据来验证模型能跑)
    print("\n 开始模拟推理测试...")
    model.eval()

    dummy_input = torch.randn(1, 1, 16000).to(device)

    with torch.no_grad():
        output = model(dummy_input)
        pred_index = output.argmax(dim=-1).item()
        pred_label = LABELS[pred_index]

    print(f" 输入数据形状: {dummy_input.shape}")
    print(f" 识别结果: {pred_label}")
    print("\n 演示完成！证明代码逻辑通顺，模型文件完好。")

run_online_demo()

 正在 Colab 云端初始化语音助手...
 运行环境: cpu
 模型加载成功！参数已读取。

 开始模拟推理测试...
 输入数据形状: torch.Size([1, 1, 16000])
 识别结果: unknown

 演示完成！证明代码逻辑通顺，模型文件完好。


In [ ]:
import torch
import torchaudio
import torch.nn.functional as F
import os

# 1. 配置 (确保这里和你训练时的标签完全一致)
LABELS = ['yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go', 'unknown']
MODEL_PATH = './voice_model.pth'
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. 定义模型 (M5) - 保持不变
class M5(torch.nn.Module):
    def __init__(self, n_input=1, n_output=11, stride=16, n_channel=32):
        super().__init__()
        self.conv1 = torch.nn.Conv1d(n_input, n_channel, kernel_size=80, stride=stride)
        self.bn1 = torch.nn.BatchNorm1d(n_channel)
        self.pool1 = torch.nn.MaxPool1d(4)
        self.conv2 = torch.nn.Conv1d(n_channel, n_channel, kernel_size=3)
        self.bn2 = torch.nn.BatchNorm1d(n_channel)
        self.pool2 = torch.nn.MaxPool1d(4)
        self.conv3 = torch.nn.Conv1d(n_channel, 2 * n_channel, kernel_size=3)
        self.bn3 = torch.nn.BatchNorm1d(2 * n_channel)
        self.pool3 = torch.nn.MaxPool1d(4)
        self.conv4 = torch.nn.Conv1d(2 * n_channel, 2 * n_channel, kernel_size=3)
        self.bn4 = torch.nn.BatchNorm1d(2 * n_channel)
        self.pool4 = torch.nn.MaxPool1d(4)
        self.fc1 = torch.nn.Linear(2 * n_channel, n_output)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(self.bn1(x))
        x = self.pool1(x)
        x = self.conv2(x)
        x = F.relu(self.bn2(x))
        x = self.pool2(x)
        x = self.conv3(x)
        x = F.relu(self.bn3(x))
        x = self.pool3(x)
        x = self.conv4(x)
        x = F.relu(self.bn4(x))
        x = self.pool4(x)
        x = F.avg_pool1d(x, x.shape[-1])
        x = x.permute(0, 2, 1)
        x = self.fc1(x)
        return F.log_softmax(x, dim=2).squeeze(1)

# 3. 预处理函数
def predict_audio_file(filename):
    if not os.path.exists(filename):
        print(f" 找不到文件: {filename}，请先上传！")
        return

    print(f"\n 正在测试文件: {filename} ...")

    # 加载音频
    waveform, sample_rate = torchaudio.load(filename)

    # A. 转单声道
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    # B. 重采样到16000Hz
    if sample_rate != 16000:
        resampler = torchaudio.transforms.Resample(sample_rate, 16000)
        waveform = resampler(waveform)

    # C. 裁剪或填充到1秒
    if waveform.shape[1] > 16000:
        waveform = waveform[:, :16000]
    else:
        padding = 16000 - waveform.shape[1]
        waveform = F.pad(waveform, (0, padding))

    # D. 预测
    tensor = waveform.unsqueeze(0).to(DEVICE) # 增加Batch维度

    # 加载模型
    model = M5(n_output=len(LABELS)).to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()

    with torch.no_grad():
        output = model(tensor)
        # 获取概率最大的那个
        pred_index = output.argmax(dim=-1).item()
        confidence = output.exp().max().item() * 100 # 算出置信度百分比

    print(f" 识别结果: 【 {LABELS[pred_index].upper()} 】 (置信度: {confidence:.1f}%)")
    print("-" * 40)

predict_audio_file("yes.m4a")


 正在测试文件: yes.m4a ...
 识别结果: 【 YES 】 (置信度: 99.7%)
----------------------------------------


In [ ]:
# 1. 确保安装必要的库
!pip install gradio torchcodec soundfile

import gradio as gr
import torch
import torchaudio
import torch.nn.functional as F
import os

# ================= 配置与模型加载 =================
LABELS = ['yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go', 'unknown']
MODEL_PATH = './voice_model.pth'
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 定义模型结构 (保持不变)
class M5(torch.nn.Module):
    def __init__(self, n_input=1, n_output=11, stride=16, n_channel=32):
        super().__init__()
        self.conv1 = torch.nn.Conv1d(n_input, n_channel, kernel_size=80, stride=stride)
        self.bn1 = torch.nn.BatchNorm1d(n_channel)
        self.pool1 = torch.nn.MaxPool1d(4)
        self.conv2 = torch.nn.Conv1d(n_channel, n_channel, kernel_size=3)
        self.bn2 = torch.nn.BatchNorm1d(n_channel)
        self.pool2 = torch.nn.MaxPool1d(4)
        self.conv3 = torch.nn.Conv1d(n_channel, 2 * n_channel, kernel_size=3)
        self.bn3 = torch.nn.BatchNorm1d(2 * n_channel)
        self.pool3 = torch.nn.MaxPool1d(4)
        self.conv4 = torch.nn.Conv1d(2 * n_channel, 2 * n_channel, kernel_size=3)
        self.bn4 = torch.nn.BatchNorm1d(2 * n_channel)
        self.pool4 = torch.nn.MaxPool1d(4)
        self.fc1 = torch.nn.Linear(2 * n_channel, n_output)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(self.bn1(x))
        x = self.pool1(x)
        x = self.conv2(x)
        x = F.relu(self.bn2(x))
        x = self.pool2(x)
        x = self.conv3(x)
        x = F.relu(self.bn3(x))
        x = self.pool3(x)
        x = self.conv4(x)
        x = F.relu(self.bn4(x))
        x = self.pool4(x)
        x = F.avg_pool1d(x, x.shape[-1])
        x = x.permute(0, 2, 1)
        x = self.fc1(x)
        return F.log_softmax(x, dim=2).squeeze(1)

# 加载模型
print("正在加载模型...")
model = M5(n_output=len(LABELS)).to(DEVICE)
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()
    print("✅ 模型加载成功！")
else:
    print("❌ 警告：没找到模型文件！请检查左侧文件列表是否有 voice_model.pth")

# ================= 核心逻辑：处理音频 =================
def process_audio(audio_path):
    if audio_path is None:
        return "请先录音或上传文件", "⚪"

    print(f"收到音频: {audio_path}")

    try:
        # 1. 读取音频
        waveform, sample_rate = torchaudio.load(audio_path)

        # 2. 预处理 (转单声道、重采样、裁剪)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        if sample_rate != 16000:
            resampler = torchaudio.transforms.Resample(sample_rate, 16000)
            waveform = resampler(waveform)
        if waveform.shape[1] > 16000:
            waveform = waveform[:, :16000]
        else:
            padding = 16000 - waveform.shape[1]
            waveform = F.pad(waveform, (0, padding))

        # 3. 预测
        tensor = waveform.unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            output = model(tensor)
            pred_index = output.argmax(dim=-1).item()
            label = LABELS[pred_index]
            confidence = output.exp().max().item()

        # 4. 生成反馈
        status_text = f"识别指令: 【 {label.upper()} 】 (置信度: {confidence:.2%})"

        visual_feedback = "⚪ 待机中"
        if label == "on":
            visual_feedback = "💡 灯亮了 (ON)"
        elif label == "off":
            visual_feedback = "🌑 灯灭了 (OFF)"
        elif label == "stop":
            visual_feedback = "🛑 停止 (STOP)"
        elif label == "go":
            visual_feedback = "🟢 前进 (GO)"
        elif label in ["up", "down", "left", "right"]:
            visual_feedback = f"🤖 机器人移动: {label.upper()}"
        elif label == "yes":
            visual_feedback = "✅ 确认 (YES)"
        elif label == "no":
            visual_feedback = "❌ 拒绝 (NO)"
        else:
            visual_feedback = "🤔 听不懂 (Unknown)"

        return status_text, visual_feedback

    except Exception as e:
        return f"发生错误: {str(e)}", "⚠️ 出错了"

# ================= 构建网页界面 =================
with gr.Blocks(title="我的语音助手") as demo:
    gr.Markdown("# 🎙️ 个人智能语音中控台")
    gr.Markdown("请上传录音，或者直接使用麦克风输入指令 (on, off, stop, go...)")

    with gr.Row():
        with gr.Column():
            audio_input = gr.Audio(sources=["microphone", "upload"], type="filepath", label="请下达指令")
            btn = gr.Button("识别指令", variant="primary")
        with gr.Column():
            result_text = gr.Textbox(label="系统日志")
            visual_output = gr.Label(value="⚪ 等待指令...", label="可视化状态面板")

    btn.click(fn=process_audio, inputs=audio_input, outputs=[result_text, visual_output])

    audio_input.change(fn=process_audio, inputs=audio_input, outputs=[result_text, visual_output])


print("正在启动网页...")
demo.launch(share=True, debug=True)

正在加载模型...
✅ 模型加载成功！
正在启动网页...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://64a11b039836a82a2a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://64a11b039836a82a2a.gradio.live


In [ ]:
# 1. 确保安装所有依赖
!pip install gradio torchcodec soundfile gtts matplotlib numpy

import gradio as gr
import torch
import torchaudio
import torch.nn.functional as F
import os
import numpy as np
import matplotlib.pyplot as plt
from gtts import gTTS
import tempfile

# ================= 1. 配置与模型 (保持不变) =================
LABELS = ['yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go', 'unknown']
MODEL_PATH = './voice_model.pth'
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 模型结构 M5
class M5(torch.nn.Module):
    def __init__(self, n_input=1, n_output=11, stride=16, n_channel=32):
        super().__init__()
        self.conv1 = torch.nn.Conv1d(n_input, n_channel, kernel_size=80, stride=stride)
        self.bn1 = torch.nn.BatchNorm1d(n_channel)
        self.pool1 = torch.nn.MaxPool1d(4)
        self.conv2 = torch.nn.Conv1d(n_channel, n_channel, kernel_size=3)
        self.bn2 = torch.nn.BatchNorm1d(n_channel)
        self.pool2 = torch.nn.MaxPool1d(4)
        self.conv3 = torch.nn.Conv1d(n_channel, 2 * n_channel, kernel_size=3)
        self.bn3 = torch.nn.BatchNorm1d(2 * n_channel)
        self.pool3 = torch.nn.MaxPool1d(4)
        self.conv4 = torch.nn.Conv1d(2 * n_channel, 2 * n_channel, kernel_size=3)
        self.bn4 = torch.nn.BatchNorm1d(2 * n_channel)
        self.pool4 = torch.nn.MaxPool1d(4)
        self.fc1 = torch.nn.Linear(2 * n_channel, n_output)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(self.bn1(x))
        x = self.pool1(x)
        x = self.conv2(x)
        x = F.relu(self.bn2(x))
        x = self.pool2(x)
        x = self.conv3(x)
        x = F.relu(self.bn3(x))
        x = self.pool3(x)
        x = self.conv4(x)
        x = F.relu(self.bn4(x))
        x = self.pool4(x)
        x = F.avg_pool1d(x, x.shape[-1])
        x = x.permute(0, 2, 1)
        x = self.fc1(x)
        return F.log_softmax(x, dim=2).squeeze(1)

# 加载模型
print("🚀 正在初始化系统...")
model = M5(n_output=len(LABELS)).to(DEVICE)
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()
    print("模型加载成功！")
else:
    print("警告：没找到模型文件！")

# ================= 2. 辅助函数 =================

# 动态彩色HTML卡片
def get_colored_status_html(label, confidence):
    color = "#333333" # 黑灰色
    text_color = "white"
    icon = "⚪"
    msg = "等待指令..."

    # 颜色
    if label == "yes":
        color = "#4CAF50" # 绿色
        icon = "✅"
        msg = "YES - 确认执行"
    elif label == "no":
        color = "#F44336" # 红色
        icon = "❌"
        msg = "NO - 拒绝操作"
    elif label == "on":
        color = "#FFC107" # 亮黄色
        text_color = "#333" # 深色字
        icon = "💡"
        msg = "ON - 灯光开启"
    elif label == "off":
        color = "#212121" # 深黑灰
        icon = "🌑"
        msg = "OFF - 灯光关闭"
    elif label == "stop":
        color = "#DC143C" # 猩红色
        icon = "🛑"
        msg = "STOP - 紧急停止"
    elif label == "go":
        color = "#2196F3" # 蓝色
        icon = "🚀"
        msg = "GO - 系统启动"
    elif label in ["up", "down", "left", "right"]:
        color = "#9C27B0" # 紫色
        icon = "🤖"
        msg = f"{label.upper()} - 移动中"
    elif label == "unknown":
        color = "#9E9E9E" # 灰色
        icon = "👂"
        msg = "没听清..."

    # HTML代码
    html_code = f"""
    <div style="
        background-color: {color};
        color: {text_color};
        padding: 25px;
        border-radius: 15px;
        text-align: center;
        box-shadow: 0 4px 15px rgba(0,0,0,0.3);
        border: 2px solid rgba(255,255,255,0.2);
        transition: background-color 0.5s ease;
    ">
        <div style="font-size: 50px; margin-bottom: 10px;">{icon}</div>
        <div style="font-size: 28px; font-weight: bold; margin-bottom: 5px;">{msg}</div>
        <div style="font-size: 14px; opacity: 0.8;">置信度: {confidence:.1%}</div>
    </div>
    """
    return html_code

def text_to_speech(text):
    if not text: return None
    try:
        tts = gTTS(text, lang='zh-cn')
        tmp_audio = tempfile.mktemp(suffix='.mp3')
        tts.save(tmp_audio)
        return tmp_audio
    except:
        return None

def plot_signal(waveform):
    fig, ax = plt.subplots(figsize=(8, 3))
    fig.patch.set_facecolor('#f9f9f9')
    ax.plot(waveform.t().numpy(), color='#3F51B5')
    ax.set_axis_off()
    plt.tight_layout()
    tmp_img = tempfile.mktemp(suffix='.png')
    plt.savefig(tmp_img, bbox_inches='tight', pad_inches=0)
    plt.close()
    return tmp_img

# ================= 3. 核心逻辑 =================
def process_audio_color(audio_path):
    if audio_path is None:
        return "请提供录音", get_colored_status_html("default", 0), None, None

    # 读取与处理
    try:
        waveform, sample_rate = torchaudio.load(audio_path)
    except:
        return "文件错误", get_colored_status_html("default", 0), None, None

    # 画图
    plot_path = plot_signal(waveform)

    # 预处理
    if waveform.shape[0] > 1: waveform = torch.mean(waveform, dim=0, keepdim=True)
    if sample_rate != 16000: waveform = torchaudio.transforms.Resample(sample_rate, 16000)(waveform)
    if waveform.shape[1] > 16000: waveform = waveform[:, :16000]
    else: waveform = F.pad(waveform, (0, 16000 - waveform.shape[1]))

    # 预测
    tensor = waveform.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        output = model(tensor)
        probs = output.exp()
        confidence = probs.max().item()
        pred_index = probs.argmax(dim=-1).item()
        label = LABELS[pred_index]

    # 阈值过滤
    if confidence < 0.4: label = "unknown"

    # 生成彩色 HTML 卡片
    html_status = get_colored_status_html(label, confidence)

    # 简单的日志文本
    log_text = f"检测到: {label.upper()} (Score: {confidence:.2f})"

    # 语音回复内容
    tts_map = {
        "yes": "确认执行。", "no": "已取消。", "on": "灯已打开。",
        "off": "灯已关闭。", "stop": "系统停止！", "go": "正在启动。",
        "unknown": None
    }
    audio_response = text_to_speech(tts_map.get(label, f"识别为{label}"))

    return log_text, html_status, plot_path, audio_response

# ================= 4. 界面构建 =================
theme = gr.themes.Soft(primary_hue="blue", radius_size="lg")

with gr.Blocks(title="AI 变色龙中控", theme=theme) as demo:
    gr.Markdown("## 🎨 多模态语音控制台 (Color UI)")
    gr.Markdown("请尝试说：**Yes (绿)**, **No (红)**, **On (黄)**, **Off (黑)**, **Stop (猩红)**")

    with gr.Row():
        # 左侧控制
        with gr.Column(scale=1):
            audio_input = gr.Audio(sources=["microphone", "upload"], type="filepath", label="🎙️ 指令输入")
            btn = gr.Button("⚡ 开始识别", variant="primary")
            result_text = gr.Textbox(label="后台日志", lines=1)

        # 右侧视觉反馈 (使用 HTML 组件实现变色卡片)
        with gr.Column(scale=1):
            # 这里的 HTML 组件会根据 Python 返回的代码变色
            visual_output = gr.HTML(
                value=get_colored_status_html("default", 0),
                label="可视化状态面板"
            )
            audio_output = gr.Audio(label="语音反馈", autoplay=True, interactive=False)

    with gr.Row():
        plot_output = gr.Image(label="波形可视化", show_label=False)

    # 绑定事件
    fn_outputs = [result_text, visual_output, plot_output, audio_output]
    btn.click(fn=process_audio_color, inputs=audio_input, outputs=fn_outputs)
    audio_input.change(fn=process_audio_color, inputs=audio_input, outputs=fn_outputs)

print("正在启动变色版...")
demo.launch(share=True, debug=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
🚀 正在初始化系统...
模型加载成功！


/tmp/ipython-input-2421418076.py:200: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="AI 变色龙中控", theme=theme) as demo:


正在启动变色版...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b3549053c8d920c2a8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
